# Visualizing Categorical Data - Bar Charts, Radar Charts and More

This is the third notebook in the Plotly tutorial series.

Categorical data represents discrete groups, labels, or classes—such as species, regions, or product types. Visualizing categorical data effectively is important for comparing data across categories, identifying patterns, and communicating relative importance or frequency.

In this notebook, we'll use Plotly to explore several techniques for effectively visualizing categorical variables, including:
- **Bar charts**: simple, fundamental and ideal for comparing quantities across categories
- **Stacked bar charts**: allows visualization of subcategory contributions within each main category, giving insight into both totals and internal composition, similar to a stacked area chart from the [previous notebook](02_lineplots_timeseries.ipynb)
- **Radar charts**: useful for visualizing multivariate data across categories in a circular, perhaps more aesthetically pleasing layout

Specifically, this notebook covers:
- Creating basic versions of the plots described above
- Scenarios in which to use different visualization types
- Customizing layout, colors, and interactivity
- Best practices for visualizations that analyze categorical data

For these visualizations, we will be using the same as the [Palmers Penguins](https://www.kaggle.com/datasets/satyajeetrai/palmer-penguins-dataset-for-eda) dataset, found in the [data directory](../data)

In [2]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels

## Bar Charts and Stacked Bar Charts

Bar charts are one of the most commonly used tools for visualizing categorical data. They allow us to compare quantities across discrete categories using the height or length of rectangular bars. Bar charts are effective when we want to highlight differences between groups or track changes in a single category across different conditions.

Stacked bar charts build on this by allowing us to visualize subcategories within each bar, which is useful for understanding composition. However, because only the bottom segment has a consistent baseline, comparisons between internal segments are more qualitative and quantitative, and must be done with care.

In [18]:
# Load in data
df = pd.read_csv('../data/penguins.csv')
df.dropna(inplace = True)
df.head()

,id,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


In [19]:
fig = px.bar(df, x='species', title="Count of Penguin Species")
fig.show()

We can further refine our bar charts by adding colors or CVD friendly fill patterns.

In [20]:
fig = px.bar(df, x='species', title="Counts of Penguin Species",

             # Adjust color according to data 
             color='species',

             # Adjust symbol fill according to data
             pattern_shape="species", 
             pattern_shape_sequence=[".", "x", "+"],

             labels={'species': 'Species',
                     'count': 'Count'},
             height=400)
fig.show()

Now you might notice that the bar chart stacks each data point on top of each other. To combine them, we can use the `px.histogram()` function in the following way instead.

In [21]:
fig = px.histogram(df, x='species', title="Counts of Penguin Species",

             # Adjust color according to data 
             color='species',

             # Adjust symbol fill according to data
             pattern_shape="species", 
             pattern_shape_sequence=[".", "x", "+"],

             labels={'species': 'Species',
                     'count': 'Count'},

             height=400,
             
             # We can add text to better compare quantitative variables
             text_auto=True)
fig.show()

Now we have analyzed the different species of the penguins, but as aforementioned, we can analyze the species of these penguins that make up each island using the stacked bar chart as follows:

In [28]:
fig = px.histogram(df, x='island', title = 'Counts of Different Species on Different Islands',
             hover_data=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm'], color='species', height=400)
fig.show()

Instead of "stacked" bar charts, we can also do "grouped" bar charts, which are very similar in concept, but look different visually.

In [ ]:
fig = px.histogram(df, x="sex", y="bill_depth_mm",
             color='species', barmode='group',
             
             # This will change what the histogram shows from a sum to an average of the bill depth, which is more important in our case
             histfunc='avg',

             height=400)
fig.show()

## Radar Charts

Radar charts, also known as spider or web charts, are useful for comparing multiple variables across different categories on a circular axis. Each axis represents a different feature, and data points are connected to form a polygon. They are especially helpful when we want to compare the shape or profile of observations across dimensions that are on a similar scale. However, radar charts can become cluttered quickly, so they are best used with a limited number of categories and variables.

Although Plotly doesn't have a built-in radar chart function, we can create a very simple radar chart on the Palmers penguins dataset as shown below:

In [34]:
# Data cleaning and preparation
radar_data = df.groupby('species')[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].mean()

# Normalization must be done since the values do not have the same units
for col in radar_data.columns:
    radar_data[col] = radar_data[col] / radar_data[col].max()
radar_data = radar_data.reset_index()
radar_data = radar_data.melt(id_vars='species', var_name='Measurement', value_name='Value')

In [36]:
fig = px.line_polar(radar_data, r='Value', theta='Measurement', color='species', line_close=True)
fig.show()

As we can see, we can highlight the differences between the species through this visualization, showing us the differences in quantitative variables for our different categories. We can make this look nicer using the following code:

In [46]:
fig = px.line_polar(radar_data, r='Value', theta='Measurement', color='species', line_close=True, 
                    title = 'Measurements of Different Penguin Species',
                    height = 700, width = 800

                            
                    )

# Fills in empty space between lines
fig.update_traces(fill='toself')
fig.show()

It is important to note that a statistical transformation was performed here. Since the quantitative variables do not have the same units, we must normalize them to obtain meaningful insights and not have one variable entirely skewing the visualization. This adds a statistical transformation layer to our visualization, as according to Wickham's *A Layered Grammar of Graphics* which is an important consideration.

One shortcoming of the radar chart is that, especially when there are many categories, it is difficult to keep track of which one is which. Even with Plotly's interactivity, the Adelie species is completely covered by the Gentoo and Chinstrap species, and we cannot get a lot of insight about it. Although visually pleasing, radar charts have their limitations.

## Final Remarks

Plotly makes it easy to construct clear and interactive categorical data visualizations:

- **Bar charts** are straightforward and effective for comparisons.
- **Stacked bar charts** add a compositional layer to categorical comparisons.
- **Radar charts** offer a compact way to compare multi-dimensional data across categories, although they require careful design for clarity.

These visualizations, especially when paired with interactivity and proper color/accessibility design, can reveal patterns that might otherwise remain hidden in raw data tables, thus highlighting the importance of visualizing qualitative data.

In the [next notebook](04_boxplots_distributions.ipynb), we will explore how to use Plotly to create histograms, box and violin plots, whiling continuing to apply the principles of effective visual communication.



## References

- [Plotly bar chart documentation](https://plotly.com/python/bar-charts/)
- [Plotly radar chart documentation](https://plotly.com/python/radar-chart/)
- *The Truthful Art: Data, Charts and Maps for Communication*, Albert Cairo
- *The Visual Display of Quantitative Information*, Edward Tufte
- *A Layered Grammar of Graphics*, Hadley Wickham